# 3. Costeo operativo y ruteo controlado

Este notebook documenta la etapa en la que los reclamos ya depurados se enriquecen con **distancia, tiempo estimado y costo operativo**. A diferencia de las notebooks previas, acá aparece una dependencia externa: las distancias y los tiempos de viaje se obtienen mediante la **Google Maps Directions API**.

Metodológicamente, esta es la instancia que operacionaliza esa integración externa a través de **caché persistente + automatización controlada de ruteo**. El objetivo no es “pegarle a la API porque sí”, sino administrar una corrida potencialmente costosa: primero se diagnostica cobertura existente, luego se decide si corresponde ejecutar una actualización y finalmente se reutilizan o regeneran los artifacts que alimentan los agregados operativos posteriores.

La reutilización de caché y el modo seguro importan por cuatro razones concretas: **cuota**, **costo**, **reproducibilidad** y **evitar llamadas redundantes** a un servicio externo. Los outputs de esta etapa enriquecen cada reclamo con métricas de viaje observables —distancia y duración estimada— y dejan preparada la base para el costeo downstream.

## Cómo leer este notebook

1. **Parámetros de ejecución**: explicitan tamaño de tanda, límites de tiempo y banderas de seguridad.
2. **Diagnóstico previo**: muestra si ya existe cobertura suficiente en caché y cuántos destinos siguen pendientes.
3. **Guardas de seguridad**: explican por qué la notebook corre en modo lectura o por qué habilita una actualización controlada.
4. **Artifacts y muestras**: permiten revisar la cobertura lograda y el impacto del costeo sobre los reclamos enriquecidos.

> **Modo seguro por defecto:** la notebook NO escribe salvo habilitación manual. Esto protege cuota de API, tiempos de ejecución y reproducibilidad.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from helpers.google_maps_cache import run_routing_automation


## 1. Parámetros de corrida y filosofía de control

Los parámetros siguientes definen una política conservadora: tandas acotadas, tiempo máximo de ejecución y escritura explícitamente deshabilitada por defecto. La idea es separar dos usos posibles de la notebook: **auditar artifacts existentes** o **forzar una actualización controlada** cuando realmente haga falta.

En otras palabras, la integración con Google Maps Directions API no se trata como una consulta efímera, sino como una adquisición externa que debe poder presupuestarse, repetirse y auditarse. Ese encuadre es el que vuelve defendible el costeo posterior.


In [2]:
BATCH_SIZE = 250
MAX_BATCHES = 4
MAX_RUNTIME_MINUTES = 20
TIMEOUT_SECONDS = 30
REFRESH_AGGREGATES = True
ALLOW_ROUTING_WRITE = False
FORCE_ROUTING_REFRESH = False

## 2. Diagnóstico de cobertura antes de decidir una nueva corrida

Antes de consultar servicios externos conviene mirar el estado de `routing_scope` y `distancias_cache`. Si la cobertura ya es completa o no quedan destinos `pending`, una nueva ejecución no agrega valor y solo aumenta costo y riesgo operativo. Por eso este diagnóstico aparece antes de cualquier intento de escritura.

Esta secuencia es importante porque el enriquecimiento por distancia/tiempo se hace a nivel de pares origen-destino reutilizables. Si esas combinaciones ya fueron resueltas, reconsultarlas degradaría la reproducibilidad de la corrida y consumiría cuota sin mejorar la información disponible para costear reclamos.


In [3]:
STATUS_ORDER = ['pending', 'ok', 'zero_results', 'api_error', 'request_error']
ROUTING_SCOPE_PATH = ROOT / 'data/intermediate/routing_scope.parquet'
DISTANCIAS_CACHE_PATH = ROOT / 'data/intermediate/distancias_cache.parquet'

def _routing_status_summary(path: Path, artifact_name: str) -> dict[str, object]:
    summary: dict[str, object] = {
        'artifact': artifact_name,
        'path': str(path.relative_to(ROOT)),
        'exists': path.exists(),
        'rows': 0,
    }
    for status in STATUS_ORDER:
        summary[status] = 0

    if not path.exists():
        return summary

    frame = pd.read_parquet(path)
    summary['rows'] = int(len(frame))
    if 'routing_status' in frame.columns:
        counts = frame['routing_status'].fillna('missing').astype(str).value_counts()
        for status in STATUS_ORDER:
            summary[status] = int(counts.get(status, 0))
        extra_statuses = sorted(status for status in counts.index if status not in STATUS_ORDER)
        if extra_statuses:
            summary['other_statuses'] = {status: int(counts[status]) for status in extra_statuses}
    if 'reclamos_count' in frame.columns:
        summary['reclamos_count_total'] = int(pd.to_numeric(frame['reclamos_count'], errors='coerce').fillna(0).sum())
    return summary

routing_scope_diagnostics = _routing_status_summary(ROUTING_SCOPE_PATH, 'routing_scope')
distancias_cache_diagnostics = _routing_status_summary(DISTANCIAS_CACHE_PATH, 'distancias_cache')
routing_diagnostics = pd.DataFrame([routing_scope_diagnostics, distancias_cache_diagnostics])
routing_scope_pending = (
    int(routing_scope_diagnostics['pending'])
    if routing_scope_diagnostics['exists']
    else None
)
routing_diagnostics

,artifact,path,exists,rows,pending,ok,zero_results,api_error,request_error,reclamos_count_total
0,routing_scope,data/intermediate/routing_scope.parquet,True,28524,0,28524,0,0,0,159030
1,distancias_cache,data/intermediate/distancias_cache.parquet,True,28524,0,28524,0,0,0,159030


## 3. Resultado de las guardas de seguridad

Esta salida resume la decisión operativa del notebook: si se ejecuta o no el ruteo, con qué justificación y bajo qué combinación de banderas. Es importante leerla como una **auditoría de control**, no solo como un detalle técnico.


In [4]:
EXECUTE_ROUTING = bool(ALLOW_ROUTING_WRITE)
ROUTING_SKIP_REASON = None

if not ALLOW_ROUTING_WRITE:
    EXECUTE_ROUTING = False
    ROUTING_SKIP_REASON = 'Modo seguro activo: ALLOW_ROUTING_WRITE = False. El notebook seguirá en modo lectura y reutilizará los artifacts existentes sin reescribir ruteo.'
elif routing_scope_pending == 0 and not FORCE_ROUTING_REFRESH:
    EXECUTE_ROUTING = False
    ROUTING_SKIP_REASON = 'No hay destinos pending en data/intermediate/routing_scope.parquet. Se reutilizan los artifacts ya resueltos y no se ejecuta una nueva corrida.'

guard_status = {
    'allow_routing_write': ALLOW_ROUTING_WRITE,
    'force_routing_refresh': FORCE_ROUTING_REFRESH,
    'routing_scope_pending': routing_scope_pending,
    'execute_routing': EXECUTE_ROUTING,
    'skip_reason': ROUTING_SKIP_REASON,
}

if ROUTING_SKIP_REASON:
    print(ROUTING_SKIP_REASON)

guard_status


Modo seguro activo: ALLOW_ROUTING_WRITE = False. El notebook seguirá en modo lectura y reutilizará los artifacts existentes sin reescribir ruteo.


{'allow_routing_write': False,
 'force_routing_refresh': False,
 'routing_scope_pending': 0,
 'execute_routing': False,
 'skip_reason': 'Modo seguro activo: ALLOW_ROUTING_WRITE = False. El notebook seguirá en modo lectura y reutilizará los artifacts existentes sin reescribir ruteo.'}

## 4. Artifacts de referencia, costeo y ruteo

Las tres tablas siguientes muestran la continuidad de la etapa: primero las referencias estructurales necesarias para costear, luego los outputs de costeo sobre reclamos y finalmente los artifacts propios del ruteo. Juntas permiten verificar qué capas quedaron listas para la agregación posterior.


In [5]:
REFERENCE_ARTIFACTS = {
    'sede_ref': ROOT / 'data/processed/sede_ref.parquet',
    'servicio_sede_ref': ROOT / 'data/processed/servicio_sede_ref.parquet',
    'costos_ref': ROOT / 'data/processed/costos_ref.parquet',
    'report': ROOT / 'reports/phase3_reference_inputs.md',
}
ROUTING_ARTIFACTS = {
    'distancias_cache': ROOT / 'data/intermediate/distancias_cache.parquet',
    'routing_scope': ROOT / 'data/intermediate/routing_scope.parquet',
    'report': ROOT / 'reports/phase3_routing_prep.md',
}
COSTING_ARTIFACTS = {
    'distancias_costos': ROOT / 'data/processed/distancias_costos.parquet',
    'reclamos_enriquecidos': ROOT / 'data/processed/reclamos_enriquecidos.parquet',
    'report': ROOT / 'reports/phase3_costing_outputs.md',
}
AGGREGATE_ARTIFACTS = {
    'resumen_operativo_diario': ROOT / 'data/processed/resumen_operativo_diario.parquet',
    'resumen_hotspots': ROOT / 'data/processed/resumen_hotspots.parquet',
    'report': ROOT / 'reports/phase3_operational_aggregates.md',
}

if EXECUTE_ROUTING:
    automation_result = run_routing_automation(
        batch_size=BATCH_SIZE,
        max_batches=MAX_BATCHES,
        max_runtime_minutes=MAX_RUNTIME_MINUTES,
        timeout_s=TIMEOUT_SECONDS,
        refresh_aggregates=REFRESH_AGGREGATES,
    )
    reference_result = automation_result['reference_result']
    routing_result = automation_result['last_batch']['routing_result']
    costing_result = automation_result['last_batch']['costing_result']
    aggregates_result = automation_result['last_batch'].get('aggregates_result')
else:
    automation_result = {
        'summary': {
            'executed': False,
            'skip_reason': ROUTING_SKIP_REASON,
            'batch_size': BATCH_SIZE,
            'max_batches': MAX_BATCHES,
            'max_runtime_minutes': MAX_RUNTIME_MINUTES,
        }
    }
    reference_result = {
        'summary': {'executed': False, 'source': 'existing_artifacts'},
        'artifacts': REFERENCE_ARTIFACTS,
    }
    routing_result = {
        'summary': {
            'executed': False,
            'source': 'existing_artifacts',
            'pending': routing_scope_pending,
            'ok_cached': int(distancias_cache_diagnostics['ok']),
            'zero_results': int(distancias_cache_diagnostics['zero_results']),
            'api_error': int(distancias_cache_diagnostics['api_error']),
            'request_error': int(distancias_cache_diagnostics['request_error']),
        },
        'artifacts': ROUTING_ARTIFACTS,
    }
    costing_result = {
        'summary': {'executed': False, 'source': 'existing_artifacts'},
        'artifacts': COSTING_ARTIFACTS,
    }
    aggregates_result = {
        'summary': {'executed': False, 'source': 'existing_artifacts'},
        'artifacts': AGGREGATE_ARTIFACTS,
    } if REFRESH_AGGREGATES else None

{
    'automation_summary': automation_result['summary'],
    'reference_summary': reference_result['summary'],
    'routing_summary': routing_result['summary'],
    'costing_summary': costing_result['summary'],
    'aggregates_summary': None if aggregates_result is None else aggregates_result['summary'],
}


{'automation_summary': {'executed': False,
  'skip_reason': 'Modo seguro activo: ALLOW_ROUTING_WRITE = False. El notebook seguirá en modo lectura y reutilizará los artifacts existentes sin reescribir ruteo.',
  'batch_size': 250,
  'max_batches': 4,
  'max_runtime_minutes': 20},
 'reference_summary': {'executed': False, 'source': 'existing_artifacts'},
 'routing_summary': {'executed': False,
  'source': 'existing_artifacts',
  'pending': 0,
  'ok_cached': 28524,
  'zero_results': 0,
  'api_error': 0,
  'request_error': 0},
 'costing_summary': {'executed': False, 'source': 'existing_artifacts'},
 'aggregates_summary': {'executed': False, 'source': 'existing_artifacts'}}

In [6]:
pd.DataFrame({
    'reference_artifact': list(reference_result['artifacts'].keys()),
    'reference_path': [str(Path(path).relative_to(ROOT)) for path in reference_result['artifacts'].values()]
})

,reference_artifact,reference_path
0,sede_ref,data/processed/sede_ref.parquet
1,servicio_sede_ref,data/processed/servicio_sede_ref.parquet
2,costos_ref,data/processed/costos_ref.parquet
3,report,reports/phase3_reference_inputs.md


In [7]:
pd.DataFrame({
    'costing_artifact': list(costing_result['artifacts'].keys()),
    'costing_path': [str(Path(path).relative_to(ROOT)) for path in costing_result['artifacts'].values()]
})

,costing_artifact,costing_path
0,distancias_costos,data/processed/distancias_costos.parquet
1,reclamos_enriquecidos,data/processed/reclamos_enriquecidos.parquet
2,report,reports/phase3_costing_outputs.md


## 5. Lectura rápida de cobertura y ejemplos enriquecidos

Para cerrar, se muestran resúmenes por `routing_status` y algunas filas de ejemplo. El propósito no es explorar patrones espaciales en profundidad, sino comprobar que la capa enriquecida ya contiene información operativa utilizable y que la cobertura del cache es consistente con el modo de ejecución adoptado.


In [8]:
pd.DataFrame({
    'routing_artifact': list(routing_result['artifacts'].keys()),
    'routing_path': [str(Path(path).relative_to(ROOT)) for path in routing_result['artifacts'].values()]
})

,routing_artifact,routing_path
0,distancias_cache,data/intermediate/distancias_cache.parquet
1,routing_scope,data/intermediate/routing_scope.parquet
2,report,reports/phase3_routing_prep.md


In [9]:
routing_scope = pd.read_parquet(routing_result['artifacts']['routing_scope'])
routing_scope[['routing_status', 'reclamos_count']].groupby('routing_status', dropna=False).agg(destinos=('reclamos_count', 'size'), reclamos=('reclamos_count', 'sum')).reset_index().sort_values('destinos', ascending=False)

,routing_status,destinos,reclamos
0,ok,28524,159030


In [10]:
reclamos_enriquecidos = pd.read_parquet(costing_result['artifacts']['reclamos_enriquecidos'])
reclamos_enriquecidos[['routing_status', 'costo_operativo_ars']].groupby('routing_status', dropna=False).agg(reclamos=('routing_status', 'size'), costo_total_ars=('costo_operativo_ars', 'sum')).reset_index().sort_values('reclamos', ascending=False)

,routing_status,reclamos,costo_total_ars
0,ok,159030,1.211976e+09


In [11]:
reclamos_enriquecidos.loc[reclamos_enriquecidos['routing_status'] == 'ok'].head(10)

,reclamo_id,numero_tramite,numero_orden,fecha_reclamo,lat,lon,estado_geo,motivo,servicio,servicio_normalizado,...,costo_operativo_ars,fuente_ruteo,api_status,error_message,requested_at,cache_version,vehiculo_tipo,costo_km_base_ars,costo_hora_base_ars,tarifa_version
0,147803-38982,147803,38982,2021-01-01 05:14:00,-26.560068,-54.784101,direct,SIN ENERGIA,Energia,ENERGIA,...,5098.5025,google_maps_directions,OK,NaN,2026-04-18T20:35:59.891599+00:00,v1,vehiculo_promedio_flota,242.5,27423.0,mar-2026
1,147805-38984,147805,38984,2021-01-01 16:46:00,-26.551452,-54.719487,direct,SIN ENERGIA,Energia,ENERGIA,...,6318.4150,google_maps_directions,OK,NaN,2026-04-18T16:48:29.936108+00:00,v1,vehiculo_promedio_flota,242.5,27423.0,mar-2026
2,147806-38985,147806,38985,2021-01-01 18:55:00,-26.567776,-54.730573,direct,SIN ENERGIA,Energia,ENERGIA,...,4031.5825,google_maps_directions,OK,NaN,2026-04-18T20:55:47.333166+00:00,v1,vehiculo_promedio_flota,242.5,27423.0,mar-2026
3,147807-38986,147807,38986,2021-01-01 20:26:00,-26.568716,-54.749879,direct,BAJA TENSION,Energia,ENERGIA,...,1446.4225,google_maps_directions,OK,NaN,2026-04-18T16:50:50.724337+00:00,v1,vehiculo_promedio_flota,242.5,27423.0,mar-2026
4,147808-38987,147808,38987,2021-01-01 20:26:00,-26.472822,-54.714013,direct,SIN ENERGIA,Energia,ENERGIA,...,13825.0175,google_maps_directions,OK,NaN,2026-04-18T20:53:55.985256+00:00,v1,vehiculo_promedio_flota,242.5,27423.0,mar-2026
5,147810-7298,147810,7298,2021-01-02 07:41:00,-26.537925,-54.649263,direct,FALTA DE AGUA,Agua Potable,AGUA POTABLE,...,11711.8125,google_maps_directions,OK,NaN,2026-04-18T23:02:18.038441+00:00,v1,vehiculo_promedio_flota,242.5,27423.0,mar-2026
6,147811-38989,147811,38989,2021-01-02 07:55:00,-26.572488,-54.733239,direct,SIN ENERGIA,Energia,ENERGIA,...,3143.2050,google_maps_directions,OK,NaN,2026-04-18T20:28:06.090722+00:00,v1,vehiculo_promedio_flota,242.5,27423.0,mar-2026
7,147812-38990,147812,38990,2021-01-02 09:00:00,-26.595006,-54.734545,direct,VARIOS,Energia,ENERGIA,...,5896.4325,google_maps_directions,OK,NaN,2026-04-18T20:24:02.402428+00:00,v1,vehiculo_promedio_flota,242.5,27423.0,mar-2026
8,147814-38992,147814,38992,2021-01-02 11:35:00,-26.510873,-54.591587,direct,SIN ENERGIA,Energia,ENERGIA,...,21760.9375,google_maps_directions,OK,NaN,2026-04-18T21:17:02.387502+00:00,v1,vehiculo_promedio_flota,242.5,27423.0,mar-2026
9,147815-12295,147815,12295,2021-01-02 12:30:00,-26.726105,-54.721433,direct,SIN SEÑAL,TV Cable,TV CABLE,...,17303.0600,google_maps_directions,OK,NaN,2026-04-18T23:02:15.803634+00:00,v1,vehiculo_promedio_flota,242.5,27423.0,mar-2026


In [12]:
reclamos_enriquecidos.loc[
    reclamos_enriquecidos["routing_status"] == "ok",
    [
        "reclamo_id",
        "lat",
        "lon",
        "distance_m",
        "distance_km",
        "duration_s",
        "duration_h",
        "costo_operativo_ars",
        "fuente_ruteo",
        "api_status",
    ]
].head(20)

,reclamo_id,lat,lon,distance_m,distance_km,duration_s,duration_h,costo_operativo_ars,fuente_ruteo,api_status
0,147803-38982,-26.560068,-54.784101,3371.0,3.371,562.0,0.156111,5098.5025,google_maps_directions,OK
1,147805-38984,-26.551452,-54.719487,6454.0,6.454,624.0,0.173333,6318.4150,google_maps_directions,OK
2,147806-38985,-26.567776,-54.730573,3212.0,3.212,427.0,0.118611,4031.5825,google_maps_directions,OK
3,147807-38986,-26.568716,-54.749879,813.0,0.813,164.0,0.045556,1446.4225,google_maps_directions,OK
4,147808-38987,-26.472822,-54.714013,18970.0,18.970,1211.0,0.336389,13825.0175,google_maps_directions,OK
5,147810-7298,-26.537925,-54.649263,13931.0,13.931,1094.0,0.303889,11711.8125,google_maps_directions,OK
6,147811-38989,-26.572488,-54.733239,2627.0,2.627,329.0,0.091389,3143.2050,google_maps_directions,OK
7,147812-38990,-26.595006,-54.734545,5562.0,5.562,597.0,0.165833,5896.4325,google_maps_directions,OK
8,147814-38992,-26.510873,-54.591587,22639.0,22.639,2136.0,0.593333,21760.9375,google_maps_directions,OK
9,147815-12295,-26.726105,-54.721433,22538.0,22.538,1554.0,0.431667,17303.0600,google_maps_directions,OK
